# Ensemble-Based Hybrid IDS: Random Forest + CNN-BiGRU (Revised)

This notebook reproduces the analysis in the revised manuscript **"An Ensemble-Based Hybrid Framework for Malicious Network Traffic Detection Using Random Forest and CNN-BiGRU Models"** on the **CIC-ToN-IoT** dataset.

It implements, in order:
1. Data loading and cleaning (removal of constant features, non-finite handling)
2. Exploratory analysis (binary and multiclass class distribution)
3. Train-only standardisation (no leakage)
4. Random-Forest-importance **feature selection** + **ablation study**
5. **Stratified k-fold cross-validation** of classical models (RF, KNN, Decision Tree, Logistic Regression) with mean ± SD and 95% CI
6. **Statistical significance testing** (paired t-test, McNemar)
7. **CNN-BiGRU** model
8. **Correct stacking ensemble** (out-of-fold RF + CNN-BiGRU probabilities → logistic-regression meta-learner)
9. **Computational-cost** measurement (train/inference time, model size, parameters)
10. **Multiclass** attack-type classification + **class-imbalance** handling
11. A clearly marked **second-dataset validation template**

> **Note on scale.** Deep-learning cross-validation on the full ~4.85M-flow dataset is expensive. Set `SUBSAMPLE` below to run quickly on a stratified subsample (rare classes preserved) or `None` to use the full data on adequate hardware.

In [ ]:
import os, time, json, warnings, pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from scipy.stats import chi2

from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             confusion_matrix, roc_curve, auc, balanced_accuracy_score,
                             precision_recall_fscore_support)

warnings.filterwarnings('ignore')
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# TensorFlow is only needed for the CNN-BiGRU / stacking sections
try:
    import tensorflow as tf
    from tensorflow.keras import layers, models
    tf.random.set_seed(RANDOM_STATE)
    TF_AVAILABLE = True
except Exception as e:
    print('TensorFlow not available; deep-learning cells will be skipped:', e)
    TF_AVAILABLE = False

## 1. Load and clean the data

In [ ]:
# Path to the CIC-ToN-IoT parquet file (77 features + Label + Attack)
DATA_PATH = 'CIC-ToN-IoT-V2.parquet'   # <-- adjust as needed
SUBSAMPLE = 250_000   # set to None to use the full dataset (needs plenty of RAM/CPU)

df = pd.read_parquet(DATA_PATH)
print('Raw shape:', df.shape)

# Standardise label column names if necessary
assert 'Label' in df.columns, 'Expected a binary Label column'
assert 'Attack' in df.columns, 'Expected an Attack (attack-type) column'

# Drop rows with a missing label
df = df.dropna(subset=['Label']).reset_index(drop=True)

feature_cols = [c for c in df.columns if c not in ('Label', 'Attack')]
print('Feature columns:', len(feature_cols))

In [ ]:
# --- Cleaning: replace non-finite values, drop exact duplicates, drop constant features ---
df[feature_cols] = df[feature_cols].replace([np.inf, -np.inf], np.nan)
df[feature_cols] = df[feature_cols].fillna(0)

before = len(df)
df = df.drop_duplicates().reset_index(drop=True)
print(f'Removed {before - len(df)} duplicate rows')

# Identify and drop zero-variance (constant) features
nunique = df[feature_cols].nunique()
constant_cols = nunique[nunique <= 1].index.tolist()
print(f'Constant (zero-variance) features dropped ({len(constant_cols)}):')
print(constant_cols)
feature_cols = [c for c in feature_cols if c not in constant_cols]
print('Informative features remaining:', len(feature_cols))

## 2. Exploratory analysis: class distribution

In [ ]:
# Binary label mapping: benign vs attack
# (In CIC-ToN-IoT the Attack column has 'Benign' plus 9 attack types.)
print('Binary distribution:')
print(df['Label'].value_counts(normalize=True).round(4))
print()
print('Attack-type distribution:')
print(df['Attack'].value_counts())

ax = df['Attack'].value_counts().plot(kind='bar', figsize=(9,3.5), color='#2c7fb8')
ax.set_ylabel('count'); ax.set_title('CIC-ToN-IoT attack-type distribution'); ax.set_yscale('log')
plt.tight_layout(); plt.show()

In [ ]:
# Optional stratified subsample (preserves rare classes) for tractable experimentation
if SUBSAMPLE is not None and len(df) > SUBSAMPLE:
    per_class = max(1, SUBSAMPLE // df['Attack'].nunique())
    df = (df.groupby('Attack', group_keys=False)
            .apply(lambda g: g.sample(min(len(g), per_class), random_state=RANDOM_STATE))
            .reset_index(drop=True))
    print('Working (subsampled) shape:', df.shape)

X_all = df[feature_cols].values.astype('float32')
y_bin = df['Label'].astype(int).values if df['Label'].dtype != object else (df['Attack'] != 'Benign').astype(int).values
print('Binary positive rate:', round(float(y_bin.mean()), 4))

## 3. Feature selection (Random-Forest importance) and ablation

RF is used here **only as an importance estimator** to rank and prune features. A separate RF is trained later as a base classifier. We keep the smallest set of top-ranked features whose cumulative importance reaches 99% (minimum 20).

In [ ]:
# Fit importance estimator on a train split (no leakage into evaluation)
Xtr, Xte, ytr, yte = train_test_split(X_all, y_bin, test_size=0.2, stratify=y_bin, random_state=RANDOM_STATE)
scaler = StandardScaler().fit(Xtr)          # fit on TRAIN ONLY
Xtr_s, Xte_s = scaler.transform(Xtr), scaler.transform(Xte)

imp_rf = RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE, n_jobs=-1).fit(Xtr_s, ytr)
importances = imp_rf.feature_importances_
order = np.argsort(importances)[::-1]
ranked = [(feature_cols[i], float(importances[i])) for i in order]

cum, k = 0.0, 0
for _, v in ranked:
    cum += v; k += 1
    if cum >= 0.99 and k >= 20:
        break
selected_idx = order[:k]
selected_features = [feature_cols[i] for i in selected_idx]
print(f'Selected {len(selected_features)} of {len(feature_cols)} features (cumulative importance {cum:.3f})')
print('Top 15:', [f'{n} ({v:.3f})' for n, v in ranked[:15]])

In [ ]:
# Ablation: accuracy vs number of top-ranked features
ks = [10, 15, 20, 25, 30, 40, len(selected_features), len(feature_cols)]
abl = []
for kk in ks:
    idx = order[:kk]
    m = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=RANDOM_STATE, n_jobs=-1)
    m.fit(Xtr_s[:, idx], ytr)
    acc = accuracy_score(yte, m.predict(Xte_s[:, idx]))
    abl.append((kk, acc))
    print(f'k={kk:>3}  acc={acc:.4f}')

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
top = ranked[:15][::-1]
ax[0].barh([t[0] for t in top], [t[1] for t in top], color='#2c7fb8')
ax[0].set_xlabel('RF Gini importance'); ax[0].set_title('Top-15 feature importances')
ax[1].plot([a[0] for a in abl], [a[1] for a in abl], 'o-', color='#d95f0e')
ax[1].axvline(len(selected_features), ls='--', color='green', label=f'selected k={len(selected_features)}')
ax[1].set_xlabel('# features'); ax[1].set_ylabel('accuracy'); ax[1].legend(); ax[1].set_title('Ablation')
plt.tight_layout(); plt.show()

# Restrict feature matrix to selected features for the remaining experiments
Xsel = X_all[:, selected_idx]

## 4. Binary detection: stratified k-fold cross-validation (classical models)

Reported as mean ± SD with 95% CI. KNN is included as a baseline (per reviewer request).

In [ ]:
def ci95(a):
    a = np.asarray(a); return 1.96 * a.std(ddof=1) / np.sqrt(len(a))

models = {
    'RandomForest': RandomForestClassifier(n_estimators=100, max_depth=10, random_state=RANDOM_STATE, n_jobs=-1),
    'KNN':          KNeighborsClassifier(n_neighbors=5, n_jobs=-1),
    'DecisionTree': DecisionTreeClassifier(max_depth=12, random_state=RANDOM_STATE),
    'LogReg':       LogisticRegression(max_iter=1000),
}

# Use a bounded sample for CV speed if very large
N = min(100_000, len(Xsel))
idx = np.random.RandomState(RANDOM_STATE).permutation(len(Xsel))[:N]
Xcv, ycv = Xsel[idx], y_bin[idx]

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
fold_acc = {m: [] for m in models}
cv_metrics = {m: {'acc': [], 'prec': [], 'rec': [], 'f1': []} for m in models}

for tr, va in skf.split(Xcv, ycv):
    sc = StandardScaler().fit(Xcv[tr])
    Xtr_f, Xva_f = sc.transform(Xcv[tr]), sc.transform(Xcv[va])
    for name, mdl in models.items():
        from sklearn.base import clone
        m = clone(mdl).fit(Xtr_f, ycv[tr]); p = m.predict(Xva_f)
        cv_metrics[name]['acc'].append(accuracy_score(ycv[va], p))
        cv_metrics[name]['prec'].append(precision_score(ycv[va], p, zero_division=0))
        cv_metrics[name]['rec'].append(recall_score(ycv[va], p, zero_division=0))
        cv_metrics[name]['f1'].append(f1_score(ycv[va], p, zero_division=0))
        fold_acc[name].append(accuracy_score(ycv[va], p))

print(f"{'Model':<14}{'Acc (mean±SD)':<22}{'95% CI':<12}{'F1':<10}")
for name in models:
    a = np.array(cv_metrics[name]['acc']); f = np.array(cv_metrics[name]['f1'])
    print(f"{name:<14}{a.mean():.4f} ± {a.std(ddof=1):.4f}      ±{ci95(a):.4f}     {f.mean():.4f}")

## 5. Statistical significance (paired t-tests over folds)

In [ ]:
base = 'RandomForest'
for name in models:
    if name == base: continue
    t, p = stats.ttest_rel(fold_acc[base], fold_acc[name])
    verdict = 'significant' if p < 0.05 else 'NOT significant'
    print(f'{base} vs {name}: t={t:.3f}, p={p:.4f}  ->  {verdict}')

## 6. CNN-BiGRU model

Convolutional feature extraction (left branch) followed by bidirectional-GRU sequence modelling (right branch), then a dense classification head.

In [ ]:
def build_cnn_bigru(input_shape, n_classes=1):
    out_units = 1 if n_classes == 1 else n_classes
    out_act   = 'sigmoid' if n_classes == 1 else 'softmax'
    loss      = 'binary_crossentropy' if n_classes == 1 else 'sparse_categorical_crossentropy'
    model = models.Sequential([
        layers.Input(shape=input_shape),
        layers.Conv1D(64, 3, activation='relu'),
        layers.MaxPooling1D(2),
        layers.Conv1D(128, 3, activation='relu'),
        layers.MaxPooling1D(2),
        layers.Bidirectional(layers.GRU(64, return_sequences=True)),
        layers.Bidirectional(layers.GRU(64)),
        layers.Flatten(),
        layers.Dense(64, activation='relu'),
        layers.Dropout(0.3),
        layers.Dense(out_units, activation=out_act),
    ])
    model.compile(optimizer='adam', loss=loss, metrics=['accuracy'])
    return model

if TF_AVAILABLE:
    demo = build_cnn_bigru((len(selected_features), 1))
    demo.summary()

## 7. Correct stacking ensemble (RF + CNN-BiGRU → logistic-regression meta-learner)

The meta-learner is trained on **out-of-fold** probability predictions of the base models to avoid leakage. We run three repeated stratified runs and report mean ± SD, then test RF vs Stacking and vs CNN-BiGRU for significance.

In [ ]:
def eval_three_models(X, y, seeds=(1, 2, 3), epochs=5, cap=20000):
    if not TF_AVAILABLE:
        print('TensorFlow unavailable - skipping.'); return None
    if len(X) > cap:
        s = np.random.RandomState(0).permutation(len(X))[:cap]; X, y = X[s], y[s]
    res = {m: {'acc': [], 'prec': [], 'rec': [], 'f1': []} for m in ['RF', 'CNN-BiGRU', 'Stacking']}
    mcnemar = None
    for seed in seeds:
        tf.random.set_seed(seed); np.random.seed(seed)
        Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, stratify=y, random_state=seed)
        sc = StandardScaler().fit(Xtr); Xtr, Xte = sc.transform(Xtr), sc.transform(Xte)
        Xtr3, Xte3 = Xtr.reshape(-1, Xtr.shape[1], 1), Xte.reshape(-1, Xte.shape[1], 1)
        # out-of-fold meta-features (single internal split)
        Xb, Xh, yb, yh = train_test_split(Xtr, ytr, test_size=0.3, stratify=ytr, random_state=seed)
        rf_b = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=seed, n_jobs=-1).fit(Xb, yb)
        cnn_b = build_cnn_bigru((Xtr.shape[1], 1)); cnn_b.fit(Xb.reshape(-1,Xtr.shape[1],1), yb, epochs=epochs, batch_size=256, verbose=0)
        meta = LogisticRegression().fit(
            np.c_[rf_b.predict_proba(Xh)[:,1], cnn_b.predict(Xh.reshape(-1,Xtr.shape[1],1), verbose=0).ravel()], yh)
        # retrain base models on full training set
        rf = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=seed, n_jobs=-1).fit(Xtr, ytr)
        cnn = build_cnn_bigru((Xtr.shape[1], 1)); cnn.fit(Xtr3, ytr, epochs=epochs, batch_size=256, verbose=0)
        p_rf, pp_rf = rf.predict(Xte), rf.predict_proba(Xte)[:,1]
        pp_cnn = cnn.predict(Xte3, verbose=0).ravel(); p_cnn = (pp_cnn > 0.5).astype(int)
        p_st = meta.predict(np.c_[pp_rf, pp_cnn])
        for nm, pr in [('RF', p_rf), ('CNN-BiGRU', p_cnn), ('Stacking', p_st)]:
            res[nm]['acc'].append(accuracy_score(yte, pr)); res[nm]['prec'].append(precision_score(yte, pr, zero_division=0))
            res[nm]['rec'].append(recall_score(yte, pr, zero_division=0)); res[nm]['f1'].append(f1_score(yte, pr, zero_division=0))
        if mcnemar is None:
            mcnemar = (int(np.sum((p_rf==yte)&(p_st!=yte))), int(np.sum((p_rf!=yte)&(p_st==yte))))
    return res, mcnemar

out = eval_three_models(Xsel, y_bin)
if out:
    res, mcnemar = out
    for nm in res:
        a = np.array(res[nm]['acc']); r = np.array(res[nm]['rec']); f = np.array(res[nm]['f1'])
        print(f'{nm:<12} acc={a.mean():.4f}±{a.std(ddof=1):.4f}  rec={r.mean():.4f}  f1={f.mean():.4f}')

In [ ]:
if out:
    t1, p1 = stats.ttest_rel(res['RF']['acc'], res['Stacking']['acc'])
    t2, p2 = stats.ttest_rel(res['RF']['acc'], res['CNN-BiGRU']['acc'])
    print(f'RF vs Stacking : t={t1:.3f}, p={p1:.4f}  ({"sig" if p1<0.05 else "not sig"})')
    print(f'RF vs CNN-BiGRU: t={t2:.3f}, p={p2:.4f}  ({"sig" if p2<0.05 else "not sig"})')
    b, c = mcnemar; stat = (abs(b-c)-1)**2/(b+c) if (b+c) else 0.0
    print(f'McNemar RF vs Stacking: b={b}, c={c}, chi2={stat:.3f}, p={1-chi2.cdf(stat,1):.4f}')

## 8. Computational cost (training time, inference time, model size, parameters)

In [ ]:
if TF_AVAILABLE:
    Xc = Xsel[:25000]; yc = y_bin[:25000]
    Xtr, Xte, ytr, yte = train_test_split(Xc, yc, test_size=0.2, stratify=yc, random_state=RANDOM_STATE)
    sc = StandardScaler().fit(Xtr); Xtr, Xte = sc.transform(Xtr), sc.transform(Xte)

    t = time.time(); rf = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=RANDOM_STATE, n_jobs=-1).fit(Xtr, ytr); rf_train = time.time()-t
    t = time.time(); _ = rf.predict(Xte); rf_inf = time.time()-t
    pickle.dump(rf, open('rf_tmp.pkl','wb')); rf_size = os.path.getsize('rf_tmp.pkl')/1e6

    cnn = build_cnn_bigru((Xtr.shape[1],1))
    t = time.time(); cnn.fit(Xtr.reshape(-1,Xtr.shape[1],1), ytr, epochs=5, batch_size=256, verbose=0); cnn_train = time.time()-t
    t = time.time(); _ = cnn.predict(Xte.reshape(-1,Xtr.shape[1],1), verbose=0); cnn_inf = time.time()-t

    print(f'RF        : train={rf_train:.2f}s  infer={rf_inf:.3f}s  size={rf_size:.2f}MB  nodes={sum(t.tree_.node_count for t in rf.estimators_)}')
    print(f'CNN-BiGRU : train={cnn_train:.2f}s  infer={cnn_inf:.3f}s  params={cnn.count_params()}')

## 9. Multiclass attack-type classification and class imbalance

The 10-class problem is far harder because of severe imbalance. We compare RF without and with balanced class weights, and include a KNN baseline.

In [ ]:
le = LabelEncoder(); y_mc = le.fit_transform(df['Attack'].values)
classes = list(le.classes_)
Xtr, Xte, ytr, yte = train_test_split(Xsel, y_mc, test_size=0.2, stratify=y_mc, random_state=RANDOM_STATE)
sc = StandardScaler().fit(Xtr); Xtr, Xte = sc.transform(Xtr), sc.transform(Xte)

def report(pred, name):
    print(f'\n[{name}]  acc={accuracy_score(yte,pred):.4f}  '
          f'macroF1={f1_score(yte,pred,average="macro"):.4f}  '
          f'weightedF1={f1_score(yte,pred,average="weighted"):.4f}  '
          f'balAcc={balanced_accuracy_score(yte,pred):.4f}')

rf_mc  = RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE, n_jobs=-1).fit(Xtr, ytr)
report(rf_mc.predict(Xte), 'RF baseline')
rf_bal = RandomForestClassifier(n_estimators=200, class_weight='balanced_subsample', random_state=RANDOM_STATE, n_jobs=-1).fit(Xtr, ytr)
report(rf_bal.predict(Xte), 'RF class-weighted')

# Per-class F1 comparison
f1a = f1_score(yte, rf_mc.predict(Xte), average=None, labels=range(len(classes)), zero_division=0)
f1b = f1_score(yte, rf_bal.predict(Xte), average=None, labels=range(len(classes)), zero_division=0)
print('\nPer-class F1 (baseline -> weighted):')
for i, c in enumerate(classes):
    print(f'  {c:<12} {f1a[i]:.3f} -> {f1b[i]:.3f}')

In [ ]:
# KNN multiclass baseline (evaluated on a subset for speed)
knn = KNeighborsClassifier(n_neighbors=5, n_jobs=-1).fit(Xtr, ytr)
sub = np.random.RandomState(0).permutation(len(yte))[:min(20000, len(yte))]
pk = knn.predict(Xte[sub])
print(f'KNN  acc={accuracy_score(yte[sub],pk):.4f}  macroF1={f1_score(yte[sub],pk,average="macro"):.4f}')

## 10. Second-dataset validation (TEMPLATE — fill in to run)

Cross-dataset validation is the primary future-work item in the paper. The pipeline above is dataset-agnostic: load a second dataset (e.g. **UNSW-NB15**, **UNSW-ToN-IoT**, **CICIDS2017** or **BoT-IoT**), align it to the same feature space, and reuse the functions above. This cell is a ready-to-complete template.

In [ ]:
# ---------------------------------------------------------------------------
# SECOND-DATASET VALIDATION TEMPLATE
# ---------------------------------------------------------------------------
# 1) Load the second dataset into a DataFrame `df2` with a binary 'Label'
#    (and optionally an 'Attack' column for multiclass).
#
# SECOND_DATA_PATH = 'UNSW-NB15.parquet'   # or CICIDS2017 / BoT-IoT / UNSW-ToN-IoT
# df2 = pd.read_parquet(SECOND_DATA_PATH)
#
# 2) Clean identically: replace inf, fill NaN, drop duplicates.
# 3) Align features: keep the intersection with `selected_features`
#    (rename columns to match CICFlowMeter naming if required), or refit
#    feature selection on df2 using the RF-importance procedure in Section 3.
#
# common = [c for c in selected_features if c in df2.columns]
# X2 = df2[common].replace([np.inf,-np.inf],0).fillna(0).values.astype('float32')
# y2 = (df2['Attack'] != 'Benign').astype(int).values
#
# 4a) Cross-dataset generalisation (train on dataset 1, test on dataset 2):
# scaler_full = StandardScaler().fit(Xsel)          # fit on dataset-1 features
# rf_full = RandomForestClassifier(n_estimators=100, max_depth=10,
#                                  random_state=RANDOM_STATE, n_jobs=-1).fit(scaler_full.transform(Xsel), y_bin)
# print('Cross-dataset acc:', accuracy_score(y2, rf_full.predict(scaler_full.transform(X2))))
#
# 4b) Within-dataset validation on dataset 2 (reuse the CV / stacking / cost cells):
# out2 = eval_three_models(X2, y2)
#
# NOTE: This template is intentionally not executed here because the second
# dataset is not bundled with this notebook.
print('Second-dataset template ready. Provide df2 and uncomment to run.')